# Linear Regression — Gradient Descent Formulation

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is the target variable.

The linear regression model assumes

$$
y_i = w_0 + \sum_{j=1}^{D} w_j x_{ij} + \epsilon_i
$$

where the noise term follows

$$
\epsilon_i \sim N(0,\sigma^2)
$$

---

# 2. Matrix Representation

Define the design matrix

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \dots & x_{1D} \\
x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \dots & \vdots \\
x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

and the target vector

$$
y =
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

The linear model becomes

$$
y = Xw + \epsilon
$$

---

# 3. Adding the Bias Term

To incorporate the intercept term we augment the design matrix with a column of ones

$$
X =
\begin{bmatrix}
1 & x_{11} & x_{12} & \dots & x_{1D} \\
1 & x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \vdots & \dots & \vdots \\
1 & x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

The weight vector becomes

$$
w =
\begin{bmatrix}
w_0 \\
w_1 \\
\vdots \\
w_D
\end{bmatrix}
$$

---

# 4. Objective Function (Mean Squared Error)

Linear regression minimizes the squared prediction error

$$
L(w) =
\frac{1}{2N}
\sum_{i=1}^{N}
(y_i - x_i^T w)^2
$$

In matrix form

$$
L(w) =
\frac{1}{2N}
||y - Xw||_2^2
$$

This loss function is **convex**, ensuring a unique global minimum when $X^TX$ is full rank.

---

# 5. Gradient of the Loss Function

To minimize the loss function we compute its gradient with respect to the weight vector.

First expand the loss

$$
L(w) =
\frac{1}{2N}
(y - Xw)^T (y - Xw)
$$

Taking the derivative with respect to $w$ gives

$$
\nabla L(w) =
-\frac{1}{N}X^T(y - Xw)
$$

which simplifies to

$$
\nabla L(w) =
\frac{1}{N}X^T(Xw - y)
$$

---

# 6. Hessian Matrix

The second derivative of the loss function is

$$
\nabla^2 L(w) =
\frac{1}{N}X^TX
$$

Define

$$
H =
\frac{1}{N}X^TX
$$

Then the gradient can be written as

$$
\nabla L(w) = Hw - b
$$

where

$$
b = \frac{1}{N}X^Ty
$$

---

# 7. Gradient Descent Update Rule

Gradient descent iteratively updates the weights using

$$
w^{(t+1)} =
w^{(t)} - \alpha \nabla L(w^{(t)})
$$

Substituting the gradient expression gives

$$
w^{(t+1)} =
w^{(t)} - \alpha (Hw^{(t)} - b)
$$

where

- $\alpha$ is the learning rate
- $t$ is the iteration index

---

# 8. Initialization

The algorithm typically starts with

$$
w^{(0)} = 0
$$

At this point

$$
\hat{y} = Xw^{(0)} = 0
$$

and the gradient becomes

$$
\nabla L(w^{(0)}) =
-\frac{1}{N}X^Ty
$$

---

# 9. Iterative Optimization

At each iteration

1. Compute predictions

$$
\hat{y} = Xw
$$

2. Compute the gradient

$$
\nabla L(w) =
\frac{1}{N}X^T(Xw - y)
$$

3. Update the weights

$$
w = w - \alpha \nabla L(w)
$$

This process continues until convergence.

---

# 10. Convergence Criterion

The algorithm stops when the gradient becomes sufficiently small

$$
||\nabla L(w)|| < \eta
$$

where

$$
\eta
$$

is a small tolerance value.

Alternatively, convergence may be determined using a fixed number of iterations.

---

# 11. Convergence Condition for Learning Rate

For gradient descent to converge, the learning rate must satisfy

$$
0 < \alpha < \frac{2}{\lambda_{\max}(H)}
$$

where

$$
\lambda_{\max}(H)
$$

is the largest eigenvalue of

$$
H = \frac{X^TX}{N}
$$

If $\alpha$ is too large:

- the algorithm diverges
- weights oscillate or explode

---

# 12. Geometric Interpretation

The loss function forms a **convex quadratic surface**

$$
L(w)
$$

Gradient descent moves in the direction of **steepest descent**

$$
-\nabla L(w)
$$

toward the global minimum.

When features are correlated, the surface becomes elongated, causing **slow convergence**.

---

# 13. Prediction

After training, predictions are computed using

$$
\hat{y} = Xw
$$

For a single observation

$$
\hat{y}_i =
w_0 +
\sum_{j=1}^{D} w_j x_{ij}
$$

---

# 14. Algorithm Summary

Initialize

$$
w^{(0)} = 0
$$

Repeat until convergence:

1. Compute gradient

$$
\nabla L(w) =
\frac{1}{N}X^T(Xw - y)
$$

2. Update weights

$$
w = w - \alpha \nabla L(w)
$$

3. Check stopping condition

$$
||\nabla L(w)|| < \eta
$$

---

# 15. Final Optimization Problem

The optimization problem solved by gradient descent is

$$
\boxed{
\min_w
\frac{1}{2N}
\sum_{i=1}^{N}
\left(
y_i -
\sum_{j=0}^{D} x_{ij} w_j
\right)^2
}
$$

In [1]:
class LR_gradient_descent:
    def __init__(self,alpha= 0.001, add_bias=True,num_iterations=1000000,eta =1e-5):
        self.alpha = alpha
        self.add_bias = add_bias
        self.num_iterations = num_iterations
        self.eta = eta

        self.weights = None
        self.mean = None



    def fit(self,X,y):
        X = np.asarray(X)
        y = np.asarray(y)
        y = y.reshape(-1)

        if np.var(X,axis=0).sum()==0:
            self.mean = np.mean(y)
            return
        
        else :
            if X.ndim==1:
                X = X.reshape(-1,1)
    
            
            N, D = X.shape


            if self.add_bias :
                X = np.hstack((np.ones((N,1)),X))
                D +=1
    

            self.weights = np.zeros(D)
            H = X.T @ X /N
            b = X.T @ y /N
            
            for i in range(self.num_iterations):

                grad_loss = H @ self.weights - b
                
                self.weights -=  self.alpha * grad_loss
  
                if i >0 and np.linalg.norm(grad_loss) <self.eta:
                    print(f"Solution converged in iteration :{i}")
                    print(f"Model Weighs : {self.weights}")
                    break


        return


    def predict(self,X):

        X = np.asarray(X)
        N = len(X)

        if self.weights is None:
            
            return np.array([self.mean]*N)
        else :

            if X.ndim==1:
                X = X.reshape(-1,1)
    
            if self.add_bias :
                X = np.hstack((np.ones((N,1)),X))
                  
            return X @ self.weights
    

## 1. Create Dataset

We generate a simple linear dataset

$$
y = 2.5x + 1 + \epsilon
$$

where $\epsilon$ is Gaussian noise.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)



N = 120

X = np.linspace(-3, 3, N).reshape(-1,1)

true_w = 2.5
true_b = 1.0

noise = np.random.normal(0, 0.3, N)

y = true_w * X[:,0] + true_b + noise

---

## 2. Compute Hessian and Eigenvalues

For linear regression the Hessian is

$$
H = \frac{X^T X}{N}
$$

The largest eigenvalue determines the stability bound

$$
\alpha < \frac{2}{\lambda_{\max}}
$$

In [3]:
X_design = np.hstack((np.ones((N,1)), X))

H = (X_design.T @ X_design) / N

eigvals = np.linalg.eigvals(H)

lambda_max = np.max(eigvals)

print("Eigenvalues:", eigvals)
print("Largest Eigenvalue:", lambda_max)

alpha_limit = 2/lambda_max

print("Maximum Stable Learning Rate:", alpha_limit)

Eigenvalues: [1.         3.05042017]
Largest Eigenvalue: 3.050420168067227
Maximum Stable Learning Rate: 0.6556473829201102


---

## 3. Choose Learning Rates

We choose two learning rates:

- Stable learning rate  
$$
\alpha_{stable} < \frac{2}{\lambda_{\max}}
$$

- Unstable learning rate  
$$
\alpha_{unstable} > \frac{2}{\lambda_{\max}}
$$

In [4]:
alpha_limit = 2 / lambda_max

alpha_good = 0.5 * alpha_limit
alpha_bad = 1.1 * alpha_limit

print("Stable learning rate:", alpha_good)
print("Unstable learning rate:", alpha_bad)

Stable learning rate: 0.3278236914600551
Unstable learning rate: 0.7212121212121212


---

## 4. Train Models Using Gradient Descent

In [5]:
model_good = LR_gradient_descent(
    alpha=alpha_good,
)

model_good.fit(X,y)

pred_good = model_good.predict(X)

loss_good = np.mean((y - pred_good)**2)

print("Final Loss (stable α):", loss_good)

Solution converged in iteration :30
Model Weighs : [1.03977886 2.50866823]
Final Loss (stable α): 0.09808327023633655


In [6]:
model_bad = LR_gradient_descent(
    alpha=alpha_bad,
    num_iterations=150
)

model_bad.fit(X,y)

pred_bad = model_bad.predict(X)

loss_bad = np.mean((y - pred_bad)**2)

print("Final Loss (unstable α):", loss_bad)

Final Loss (unstable α): 1.0904856034359169e+25


---

## 5. Observation
The loss for the unstable learning rate is **significantly higher** hence the model fails to converge and the predictions diverge.

Thus, the experiment confirms the theoretical convergence condition:

$$
\alpha < \frac{2}{\lambda_{\max}}
$$

where the learning rate must be below the threshold determined by the largest eigenvalue of the Hessian matrix for gradient descent to converge.

---